# What a card reads about one element

A card shows what the diagram holds at once and reads the rest when it opens. Each cell asks the read a card asks.

## Setup: a small graph of protocol 1.2

In [ ]:
from __future__ import annotations

import asyncio
from typing import Any

from maxitor.front.model.diagrams.actions.get_element_card_action import (
    ElementNotInGraphError,
    GetElementCardAction,
    GetElementCardParams,
)
from maxitor.front.model.diagrams.resources.duckdb_graph_resource import DUCKDB_GRAPH_CONNECTION_KEY, DuckDBGraphResource

from aoa.action_machine.model import BaseState

LINE = "shop.Line"
GRAPH: dict[str, Any] = {
    "schema_version": "1.2",
    "nodes": [
        {"id": LINE, "type": "Entity", "label": "Line", "properties": {"description": "One order line", "entity_kind": "core", "entity_kind_label": "Core"}},
        {"id": f"{LINE}:quantity", "type": "EntityField", "label": "quantity", "properties": {"field_type": "int", "primary_key_hint": False, "description": "How many units"}},
        {"id": f"{LINE}:id", "type": "EntityField", "label": "id", "properties": {"field_type": "str", "primary_key_hint": True}},
        {"id": f"{LINE}:lifecycle:status", "type": "Lifecycle", "label": "status", "properties": {"field_name": "status", "description": "Where the line stands"}},
    ],
    "edges": [],
}


async def card(node_id: str) -> str:
    """What a card reads for one node, in one line."""
    connections: Any = {DUCKDB_GRAPH_CONNECTION_KEY: DuckDBGraphResource.build_from_json(GRAPH)}
    action = GetElementCardAction()
    try:
        read = await action.card_summary(GetElementCardParams(id=node_id), BaseState(), None, connections)  # type: ignore[arg-type]
    except ElementNotInGraphError as refused:
        return f"refused: {refused}"
    lifecycle = f" (lifecycle {read.lifecycle_id})" if read.lifecycle_id else ""
    return f"{read.type}: {read.description}{lifecycle}"

## A table

The case: a table: its kind and description.

In [ ]:
print(await card(LINE))

entity: One order line


## A field

The case: a field: its description, which needs protocol 1.2.

In [ ]:
print(await card(f"{LINE}:quantity"))

entity_field: How many units


## A lifecycle

The case: a lifecycle names itself, which is how a card knows to draw a miniature.

In [ ]:
print(await card(f"{LINE}:lifecycle:status"))

lifecycle: Where the line stands (lifecycle shop.Line:lifecycle:status)


## A field with no description

The case: a field the model left undescribed: the card leaves the line out.

In [ ]:
print(await card(f"{LINE}:id"))

entity_field: None


## Not in this version

The case: an element this version does not hold is refused; in Diff the card then asks the older version.

In [ ]:
print(await card("shop.Gone"))

refused: not found
